# Nemotron v7.8 — Unsloth-based training (fork of 0.85 LB notebook)

**Strategy:** stop fighting HF + trust_remote_code + transformers + torchvision + cutlass.
Use Unsloth's `FastLanguageModel` (same as the 0.85 notebook) — it ships its own model
class, has fused kernels, and bypasses every import-hell issue we hit.

**Phase 1 (this notebook):** match 0.85 exactly on our data → expected LB ≥ 0.85.
**Phase 2 (later):** add stratified ordering on top — the one thing they don't have.

Differences from the 0.85 reference notebook:
  - We use our 9 category JSONL files, not their pre-tokenized corpus
  - We apply the chat template + label-mask everything except the assistant response
  - Single epoch, NUM_STEPS auto-computed from data size
  - Modal glue stripped — Kaggle only


In [ ]:
# ── Shared config ────────────────────────────────────────────────
LORA_RANK    = 32
LORA_ALPHA   = 32          # match 0.85 baseline first; bump after we beat it
LORA_DROPOUT = 0.0

MAX_SEQ_LEN       = 8192   # Nemotron eval cap; we'll truncate longer samples
MICRO_BATCH_SIZE  = 4      # per-forward batch (memory-bound)
BATCH_SIZE        = 32     # effective batch via grad-accum (32 / 4 = 8 micro-steps)
LEARNING_RATE     = 2e-4
NUM_EPOCHS        = 1      # 0.85 notebook: 1 pass through data — that's the baseline

MOE_TIE_WEIGHTS   = True   # Tinker-style tied LoRA across 128 experts
SHUFFLE_DATASET   = False  # keep category-grouped order for now (Phase 2: stratified)

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "up_proj", "down_proj",
    "in_proj", "out_proj",
    "lm_head",
]

CATEGORY_FILES = [
    "train_cot_bit_manipulation.jsonl",
    "train_cot_cipher.jsonl",
    "train_cot_cryptarithm_deduce.jsonl",
    "train_cot_cryptarithm_guess.jsonl",
    "train_cot_equation_numeric_deduce.jsonl",
    "train_cot_equation_numeric_guess.jsonl",
    "train_cot_gravity.jsonl",
    "train_cot_numeral.jsonl",
    "train_cot_unit_conversion.jsonl",
]

DATA_DIR_CANDIDATES = [
    "/kaggle/input/nemotron-categorical-splits",
    "/kaggle/input/nemotron-categorical-splits/all_categorical_splits",
    "/kaggle/input/all-categorical-splits",
    "/kaggle/input/all-categorical-splits/all_categorical_splits",
    "/kaggle/input/nemotron-cot-categorical/all_categorical_splits",
    "/kaggle/input/nemotron-cot-categorical",
]


In [ ]:
import os

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
print(f"IS_KAGGLE={IS_KAGGLE}")


In [ ]:
# ── Install Unsloth + Blackwell mamba wheels (offline) ──────────────────
# Pattern from the working v5 notebook: install to /kaggle/working/packages
# with --target so we never write to the read-only Kaggle utility-script dir.
# Also use --no-deps so pip doesn't try to upgrade Kaggle's bundled torch/etc.
if IS_KAGGLE:
    import subprocess, sys
    from pathlib import Path

    TARGET_DIR = "/kaggle/working/packages"
    PKGS_DIR   = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    WHEELS_DIR = "/kaggle/input/datasets/mayukh18/nemotron-packages"
    os.makedirs(TARGET_DIR, exist_ok=True)
    if TARGET_DIR not in sys.path:
        sys.path.insert(0, TARGET_DIR)

    # core deps (offline) — --target + --no-deps avoids read-only-fs crash
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index", "--find-links", PKGS_DIR,
        "--target", TARGET_DIR, "--no-deps",
        "unsloth", "unsloth_zoo", "trl", "peft", "transformers", "datasets",
        "accelerate", "bitsandbytes", "cut-cross-entropy",
    ])

    # Blackwell mamba CUDA wheels
    for whl in [
        f"{WHEELS_DIR}/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
        f"{WHEELS_DIR}/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
    ]:
        if os.path.exists(whl):
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "-q",
                "--target", TARGET_DIR, "--no-deps", whl,
            ])

    # optional support wheels
    for _wd in ["/kaggle/input/datasets/llkh0a/rtx-wheels/wheels"]:
        if os.path.isdir(_wd):
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q",
                 "--no-index", "--find-links", _wd,
                 "--target", TARGET_DIR, "--no-deps",
                 "protobuf==6.33.5", "sentencepiece", "safetensors", "huggingface_hub"],
                check=False,
            )

    # resolve any .pth files (namespace pkgs) our installs dropped
    for pth in Path(TARGET_DIR).glob("*.pth"):
        with pth.open() as fp:
            rel = fp.read().strip()
            p = pth.parent / rel
            if p.exists() and str(p) not in sys.path:
                sys.path.append(str(p))

    print("[ok] installs complete -> /kaggle/working/packages")


In [ ]:
def run_training() -> None:
    """Unsloth-based training on our 9 category JSONLs. One epoch, MoE tied LoRA, CCE."""
    import gc, json, math, sys, time, zipfile
    from pathlib import Path

    from unsloth import FastLanguageModel
    import kagglehub
    import torch
    from cut_cross_entropy import linear_cross_entropy
    from peft import LoraConfig
    from peft.tuners.lora import Linear as LoraLinear
    from transformers import AutoTokenizer

    # ── GPU + mamba kernel sanity ────────────────────────────────────
    import causal_conv1d, mamba_ssm
    cc = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}, sm_{cc[0]*10+cc[1]}")
    print(f"torch={torch.__version__}, cuda={torch.version.cuda}")
    print(f"mamba_ssm={mamba_ssm.__version__}, causal_conv1d={causal_conv1d.__version__}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    from causal_conv1d import causal_conv1d_fn
    _x = torch.randn(1, 256, 32, device="cuda", dtype=torch.bfloat16)
    _w = torch.randn(256, 4, device="cuda", dtype=torch.bfloat16)
    causal_conv1d_fn(_x, _w, None, activation="silu")
    print("causal_conv1d CUDA kernel: OK")

    # ── Locate model + data ──────────────────────────────────────────
    MODEL_PATH = kagglehub.model_download(
        "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
    )

    data_dir = None
    for c in DATA_DIR_CANDIDATES:
        if c and os.path.isdir(c) and any(os.path.exists(os.path.join(c, f)) for f in CATEGORY_FILES):
            data_dir = c; break
    assert data_dir, f"no data dir found; searched: {DATA_DIR_CANDIDATES}"
    print(f"Data dir: {data_dir}")

    # ── Load category JSONLs ─────────────────────────────────────────
    raw_records = []
    for fname in CATEGORY_FILES:
        fpath = os.path.join(data_dir, fname)
        if not os.path.exists(fpath):
            print(f"  [skip] {fname}"); continue
        cat = fname.replace("train_cot_", "").replace(".jsonl", "")
        n = 0
        with open(fpath) as f:
            for line in f:
                if not line.strip(): continue
                rec = json.loads(line)
                rec.setdefault("category", cat)
                raw_records.append(rec); n += 1
        print(f"  {n:>5} from {fname}")
    print(f"Total records: {len(raw_records)}")

    # ── Tokenizer + build (tokens, targets, weights) with response-only loss mask ──
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    examples: list[dict] = []
    skipped_empty = 0
    skipped_no_assistant = 0
    truncated = 0

    for rec in raw_records:
        msgs = [m for m in rec["messages"] if m["role"] != "system"]
        if not msgs or msgs[-1]["role"] != "assistant":
            skipped_no_assistant += 1; continue
        prompt_msgs = msgs[:-1]
        full_msgs   = msgs

        try:
            prompt_text = tokenizer.apply_chat_template(
                prompt_msgs, tokenize=False, add_generation_prompt=True
            )
            full_text = tokenizer.apply_chat_template(
                full_msgs, tokenize=False, add_generation_prompt=False
            )
        except Exception:
            # fallback ChatML
            prompt_text = (
                f"<|im_start|>user\n{prompt_msgs[0]['content']}<|im_end|>\n"
                f"<|im_start|>assistant\n"
            )
            full_text = prompt_text + msgs[-1]["content"] + "<|im_end|>"

        prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
        full_ids   = tokenizer(full_text,   add_special_tokens=False)["input_ids"]

        if len(full_ids) <= len(prompt_ids):
            skipped_empty += 1; continue

        # Tail-truncate: prefer keeping the assistant tail (the answer/conclusion)
        if len(full_ids) > MAX_SEQ_LEN:
            # keep last MAX_SEQ_LEN tokens; recompute prompt-portion length within window
            cut = len(full_ids) - MAX_SEQ_LEN
            full_ids = full_ids[cut:]
            new_prompt_len = max(0, len(prompt_ids) - cut)
            prompt_len_in_window = new_prompt_len
            truncated += 1
        else:
            prompt_len_in_window = len(prompt_ids)

        mask = [0] * prompt_len_in_window + [1] * (len(full_ids) - prompt_len_in_window)
        if not any(mask):
            skipped_empty += 1; continue

        # tokens / targets / weights are shifted (next-token prediction)
        examples.append({
            "category":  rec["category"],
            "tokens":    full_ids[:-1],
            "targets":   full_ids[1:],
            "weights":   [float(m) for m in mask[1:]],
        })

    print(f"\nBuilt {len(examples)} training examples")
    print(f"  truncated to {MAX_SEQ_LEN}: {truncated}")
    print(f"  skipped (empty/no-assistant): {skipped_empty + skipped_no_assistant}")

    total_unmasked = sum(sum(e["weights"]) for e in examples)
    total_tokens   = sum(len(e["tokens"]) for e in examples)
    print(f"  tokens={total_tokens:,}  unmasked={total_unmasked:,.0f}  "
          f"({100*total_unmasked/total_tokens:.1f}% of tokens contribute to loss)")

    # ── Load base model via Unsloth ─────────────────────────────────
    gc.collect(); torch.cuda.empty_cache()
    model, _ = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False, load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=True,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )

    # ── Wrap LoRA ────────────────────────────────────────────────────
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK, target_modules=TARGET_MODULES,
        lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias="none", use_gradient_checkpointing="unsloth", random_state=42,
    )
    FastLanguageModel.for_training(model)

    # ── Force Mamba CUDA fast path ───────────────────────────────────
    nemotron_mod = None
    for _name, _m in sys.modules.items():
        if "modeling_nemotron_h" in _name and hasattr(_m, "is_fast_path_available"):
            nemotron_mod = _m; break
    assert nemotron_mod is not None, "nemotron_h module not found"
    nemotron_mod.is_fast_path_available = True
    print("Mamba fast path: ENABLED")

    # ── Add LoRA to lm_head (Unsloth drops it for MoE) ──────────────
    _causal = model
    while hasattr(_causal, "model"): _causal = _causal.model
    if not isinstance(_causal.lm_head, LoraLinear):
        cfg = LoraConfig(r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT)
        model.base_model._create_and_replace(
            cfg, "default", target=_causal.lm_head, target_name="lm_head", parent=_causal,
        )
        print("Added LoRA to lm_head")

    # ── Cast LoRA params to fp32 (base bf16, MoE router fp32 by design) ──
    for name, p in model.named_parameters():
        if ".lora_" in name:
            p.data = p.data.to(torch.float32)
    for name, p in model.named_parameters():
        if ".lora_" in name:
            assert p.dtype == torch.float32, f"{name} expected fp32"
            continue
        if ".mixer.gate." in name:
            assert p.dtype == torch.float32, f"router {name} expected fp32"
            continue
        assert p.dtype == torch.bfloat16, f"{name} expected bf16, got {p.dtype}"
    print("dtype check: LoRA fp32, base bf16, router fp32 ✓")

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Model: {trainable:,} trainable / {total:,} total")

    # ── CCE forward patch ────────────────────────────────────────────
    _base = model
    while hasattr(_base, "model"): _base = _base.model

    def _patched_forward(input_ids=None, attention_mask=None, labels=None, **kw):
        out = _base.backbone(
            input_ids=input_ids, attention_mask=attention_mask,
            **{k: v for k, v in kw.items()
               if k in ("position_ids", "past_key_values", "use_cache")},
        )
        h = out[0]
        lh = _base.lm_head
        base_w = lh.base_layer.weight
        lA = lh.lora_A["default"].weight
        lB = lh.lora_B["default"].weight
        sc = lh.scaling["default"]
        lm_w = base_w + sc * lB @ lA
        if labels is not None:
            per_tok = linear_cross_entropy(h, lm_w, labels, reduction="none")
            loss = per_tok.mean()
        else:
            per_tok = None; loss = None
        model._cached_per_token_ce = per_tok
        return loss

    _base.forward = _patched_forward
    print("CCE forward patched")

    # ── MoE tied LoRA ────────────────────────────────────────────────
    moe_tied = []
    if MOE_TIE_WEIGHTS:
        w1_names = ("gate_up_proj", "up_proj", "gate_proj", ".w1.")
        w2_names = ("down_proj", ".w2.")
        for name, p in model.named_parameters():
            if not p.requires_grad or ".experts." not in name or ".lora_" not in name:
                continue
            is_w1 = any(s in name for s in w1_names)
            is_w2 = any(s in name for s in w2_names)
            is_A = ".lora_A." in name; is_B = ".lora_B." in name
            tie = (is_w1 and is_A) or (is_w2 and is_B)
            if tie and p.dim() >= 2 and p.shape[0] > 1:
                moe_tied.append(p)
        with torch.no_grad():
            for p in moe_tied:
                m = p.data.mean(dim=0, keepdim=True)
                p.data.copy_(m.expand_as(p.data))
        print(f"MoE tied params: {len(moe_tied)}")

    def _tie_grads():
        if not moe_tied: return
        with torch.no_grad():
            for p in moe_tied:
                if p.grad is None: continue
                g = p.grad.sum(dim=0, keepdim=True)
                p.grad.copy_(g.expand_as(p.grad))

    # ── Training loop ────────────────────────────────────────────────
    gc.collect(); torch.cuda.empty_cache()
    device = next(model.parameters()).device

    indices = list(range(len(examples)))
    if SHUFFLE_DATASET:
        import random as _r
        _r.Random(0).shuffle(indices)

    max_steps = (len(examples) // BATCH_SIZE) * NUM_EPOCHS
    print(f"Training: {max_steps} steps, micro={MICRO_BATCH_SIZE}, batch={BATCH_SIZE}, lr={LEARNING_RATE}")

    optimizer = None
    step = 0
    log_lines = []

    for epoch in range(NUM_EPOCHS):
        for batch_start in range(0, len(indices), BATCH_SIZE):
            if step >= max_steps: break
            bi = indices[batch_start:batch_start + BATCH_SIZE]
            if len(bi) < BATCH_SIZE: break
            batch = [examples[i] for i in bi]

            n_accum = math.ceil(len(batch) / MICRO_BATCH_SIZE)
            tot_loss_sum = 0.0; tot_w = 0.0

            for ms in range(0, len(batch), MICRO_BATCH_SIZE):
                mb = batch[ms:ms + MICRO_BATCH_SIZE]
                n = len(mb)
                ml = max(len(e["tokens"]) for e in mb)
                pi = torch.zeros(n, ml, dtype=torch.long, device=device)
                pt = torch.zeros(n, ml, dtype=torch.long, device=device)
                pw = torch.zeros(n, ml, dtype=torch.float32, device=device)
                am = torch.zeros(n, ml, dtype=torch.long, device=device)
                for i, e in enumerate(mb):
                    L = len(e["tokens"])
                    pi[i, :L] = torch.tensor(e["tokens"], dtype=torch.long)
                    pt[i, :L] = torch.tensor(e["targets"], dtype=torch.long)
                    pw[i, :L] = torch.tensor(e["weights"], dtype=torch.float32)
                    am[i, :L] = 1

                t0 = time.time()
                with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                    model(input_ids=pi, attention_mask=am, labels=pt, use_cache=False)
                    per_tok = model._cached_per_token_ce
                    weighted = per_tok * pw
                    ws = pw.sum(); ls = weighted.sum()
                    loss = ls / ws if ws > 0 else ls * 0.0
                (loss / n_accum).backward()
                tot_loss_sum += ls.item(); tot_w += ws.item()
                del loss, per_tok, weighted

            if optimizer is None:
                optimizer = torch.optim.AdamW(
                    [p for p in model.parameters() if p.requires_grad],
                    lr=LEARNING_RATE, betas=(0.9, 0.95), eps=1e-8, weight_decay=0.0,
                )
            lr = LEARNING_RATE * (1 - step / max_steps)
            for pg in optimizer.param_groups: pg["lr"] = lr
            _tie_grads()
            gn = torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], max_norm=1.0
            )
            optimizer.step(); optimizer.zero_grad()
            mean_loss = tot_loss_sum / tot_w if tot_w > 0 else 0.0
            step += 1
            peak = torch.cuda.max_memory_allocated() / 1e9
            msg = (f"step {step}/{max_steps} loss={mean_loss:.4f} "
                   f"grad_norm={gn:.3f} lr={lr:.2e} peak={peak:.1f}GB")
            print(msg, flush=True); log_lines.append(msg)

    print(f"\nDone. Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

    # ── Save adapter + rename lm_head keys ───────────────────────────
    from safetensors.torch import load_file, save_file
    save_dir = "."
    for f in os.listdir(save_dir):
        if f.startswith("adapter"):
            os.remove(os.path.join(save_dir, f))
    model.save_pretrained(save_dir)
    st = os.path.join(save_dir, "adapter_model.safetensors")
    tensors = load_file(st)
    renamed = {
        k.replace("base_model.model.lm_head.", "base_model.model.backbone.lm_head."): v
        for k, v in tensors.items()
    }
    save_file(renamed, st)

    # cleanup unsloth compile cache
    import shutil as _sh
    if os.path.isdir("unsloth_compiled_cache"):
        _sh.rmtree("unsloth_compiled_cache")

    # zip submission
    adapter_files = [f for f in os.listdir(save_dir) if f.startswith("adapter")]
    with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
        for f in adapter_files:
            zf.write(os.path.join(save_dir, f), f)
    for f in adapter_files:
        os.remove(os.path.join(save_dir, f))

    with open("training_log.txt", "w") as f:
        f.write("\n".join(log_lines))

    print("Wrote submission.zip + training_log.txt")


In [ ]:
if IS_KAGGLE:
    run_training()
